In [13]:
# Load env variables
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [14]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [15]:
# Helpers to write history
def add_user_message(messages, text):
	user_message = {"role": "user", "content": text}
	messages.append(user_message)


def add_assistant_message(messages, text):
	assistant_message = {"role": "assistant", "content": text}
	messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
	params = {
			"model": model,
			"max_tokens": 600,
			"messages": messages,
			"temperature": temperature,
			"stop_sequences": stop_sequences,
	}
	if system:
			params["system"] = system

	message = client.messages.create(**params)
	return message.content[0].text

In [ ]:
import json

def generate_dataset():
	prompt = """
	Generate an evaluation dataset for a prompt evaluation. 
	The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks.
	Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

	Example output:
		```json
				[
				{
						"task": "Description of task",
				},
			...additional
			]
		```

	* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
	* Focus on tasks that do not require writing much code

	Please generate 3 objects.
"""
	messages = []
	add_user_message(messages, prompt)
	add_assistant_message(messages, "```code")
	text = chat(messages, stop_sequences=["```"])
	return json.loads(text)

In [17]:
dataset = generate_dataset()
print(dataset)  

[{'task': "Write a Python function that extracts the AWS account ID from an IAM ARN string like 'arn:aws:iam::123456789012:user/john'", 'language': 'Python', 'complexity': 'beginner'}, {'task': "Create a JSON object that represents an AWS S3 bucket policy allowing public read access to all objects with the prefix 'public/'", 'language': 'JSON', 'complexity': 'intermediate'}, {'task': 'Write a regex pattern that matches valid AWS EC2 instance IDs (format: i- followed by 17 hexadecimal or alphanumeric characters)', 'language': 'Regex', 'complexity': 'beginner'}]


In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then retruns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [20]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [ ]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

[
  {
    "output": "# Solution\n\nHere's a Python function that extracts the AWS account ID from an IAM ARN string:\n\n```python\ndef extract_account_id(arn: str) -> str:\n    \"\"\"\n    Extracts the AWS account ID from an IAM ARN string.\n    \n    Args:\n        arn: IAM ARN string in format 'arn:aws:iam::123456789012:user/john'\n    \n    Returns:\n        The AWS account ID as a string\n    \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    parts = arn.split(':')\n    \n    if len(parts) < 5:\n        raise ValueError(f\"Invalid ARN format: {arn}\")\n    \n    account_id = parts[4]\n    \n    if not account_id or not account_id.isdigit():\n        raise ValueError(f\"Invalid or missing account ID in ARN: {arn}\")\n    \n    return account_id\n\n\n# Test cases\nif __name__ == \"__main__\":\n    # Basic test\n    arn = 'arn:aws:iam::123456789012:user/john'\n    print(f\"ARN: {arn}\")\n    print(f\"Account ID: {extract_account_id(arn)}\")  # Output: 123

In [27]:
import ast
import re
import json

In [28]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

In [33]:
def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

In [34]:
def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

In [31]:
def grade_syntax(output, test_case):
    if format == "python":
        return validate_python(output)
    elif format == "json":
        return validate_json(output)
    elif format == "regex":
        return validate_regex(output)
    else:
        return 0

In [35]:

print(validate_python("def hello():\n    return 1"))   # → 10
print(validate_python("this is not python code"))       # → 0
print(validate_json('{"key": "value"}'))                # → 10
print(validate_json("not json at all"))                 # → 0
print(validate_regex(r"^arn:[a-z]+"))                   # → 10
print(validate_regex(r"^arn:[a-z+("))                   # → 0


10
0
10
0
10
0
